# ACA 1 – Computación en la Nube
## Asistente Cloud de Atención Comercial mediante Búsqueda Semántica (Pilar Ventas)

**Especialización en Inteligencia Artificial – CUN** · Docente: Hamilton Rivera Flor
**Estudiante:** Álvaro de Jesús González Fernández

> ⚠️ **Estado:** esqueleto inicial. Ninguna celda se ha ejecutado todavía. Los resultados, métricas y conclusiones se completan solo con evidencia real de ejecución.

### 1. Presentación del proyecto

**Problema.** Los clientes de una tienda de comercio electrónico consultan en lenguaje natural sobre devoluciones, garantías, envíos y uso de productos. La información está dispersa en documentos extensos, y la búsqueda por palabras clave falla cuando el cliente no usa los términos exactos del documento.

**Objetivo general.** Diseñar e implementar un pipeline cloud de ingesta, chunking, generación de embeddings e indexación en una base de datos vectorial gestionada (Qdrant Cloud). El pipeline integra un dataset estructurado de reseñas de productos con un corpus de documentos comerciales y permite recuperar semánticamente los fragmentos más relevantes (Top-k) con su fuente y su puntuación de similitud.

**Objetivos específicos.**
1. Aprovisionar una colección vectorial en Qdrant Cloud con métrica coseno y credenciales gestionadas por variables de entorno.
2. Ingestar y validar el dataset *Consumer Reviews of Amazon Products* y el corpus documental (3 documentos simulados + 1 manual real).
3. Fragmentar los documentos con una estrategia justificada y conservar sus metadatos.
4. Generar embeddings con Gemini Embedding 2 (`gemini-embedding-2`), diferenciando documentos y consultas (recuperación asimétrica), e indexarlos por lotes sin exceder la capa gratuita.
5. Validar la recuperación con un conjunto de preguntas de prueba y analizar cuantitativamente las puntuaciones de similitud coseno.

**Alcance.** Recuperación semántica (componente *Retrieval* de un sistema RAG). La generación de respuestas con un LLM queda fuera del alcance obligatorio y se plantea solo como mejora futura.

**Arquitectura.**
```
Kaggle (reseñas CSV) ─┐
Corpus (MD + PDF) ────┴─► Colab: limpieza → validación → chunking
                                       │
                     Gemini Embedding 2 (formato documento, 768 dim)
                                       │  lotes
                          Qdrant Cloud Free (colección, Cosine)
                                       ▲
Pregunta ─► Gemini Embedding 2 (formato consulta) ─► Top-k + score + fuente
```

**Fuentes de datos:**
- Corpus D1–D3: **material académico simulado** (tienda ficticia NovaLectura).
- Manual D4: documento real del fabricante.
- Reseñas: dataset público de Kaggle.

## 2. Preparación del entorno

### 2.1 Instalación de dependencias

In [ ]:
%pip install -q qdrant-client google-genai kagglehub pandas numpy pypdf langchain-text-splitters python-dotenv requests

### 2.2 Importaciones y registro de versiones
Se registran las versiones instaladas para documentar la reproducibilidad.

In [ ]:
import os, re, time, uuid, json, glob, hashlib
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd

for pkg in ["qdrant-client", "google-genai", "kagglehub", "pandas", "numpy", "pypdf", "langchain-text-splitters"]:
    try:
        print(f"{pkg}=={version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg}: NO INSTALADO")

### 2.3 Gestión de secretos
Las credenciales **nunca** se escriben en el notebook. El orden de carga es el siguiente:
1. En Google Colab: **Secrets** (icono de llave en la barra lateral).
2. En entorno local: archivo `.env`, que no se versiona (ver `.env.example`).

La celda solo informa si cada variable existe. **Nunca imprime su valor.**

In [ ]:
REQUIRED_VARS = ["QDRANT_URL", "QDRANT_API_KEY", "GEMINI_API_KEY", "KAGGLE_USERNAME", "KAGGLE_KEY"]

try:
    from google.colab import userdata  # Entorno Colab
    for var in REQUIRED_VARS:
        try:
            value = userdata.get(var)
            if value:
                os.environ[var] = value
        except Exception:
            pass
    print("Entorno: Google Colab (Secrets)")
except ImportError:
    from dotenv import load_dotenv     # Entorno local
    load_dotenv()
    print("Entorno: local (.env)")

missing = [v for v in REQUIRED_VARS if not os.environ.get(v)]
for v in REQUIRED_VARS:
    print(f"{v}: {'OK' if v not in missing else 'FALTA'}")
assert not missing, f"Faltan variables de entorno: {missing}"

### 2.4 Configuración del proyecto
Parámetros centralizados. `EMBED_MODEL` es el código estable `gemini-embedding-2` según la documentación oficial de Google. **`EMBED_TPM_LIMIT` debe ajustarse al límite de tokens por minuto que muestre AI Studio → Límite de frecuencia.**

In [ ]:
# --- Embeddings ---
EMBED_MODEL = "gemini-embedding-2"           # Código estable (documentación oficial de Gemini API)
EMBED_DIM = 768                              # Dimensión reducida (MRL) para ahorrar almacenamiento
EMBED_BATCH = 20                             # Textos por solicitud
EMBED_TPM_LIMIT = 25000                      # Tokens/minuto a no superar (AJUSTAR al límite de AI Studio, con margen)
EMBED_PAUSE_S = 1.0                          # Pausa mínima entre solicitudes
# gemini-embedding-2 no acepta task_type: la tarea se indica con prefijos en el texto (documentación oficial)
QUERY_PREFIX = "task: question answering | query: "
DOC_TEMPLATE = "title: none | text: {text}"

# --- Base vectorial ---
COLLECTION = "aca1_ventas"
RECREATE_COLLECTION = False                  # True solo para reconstruir el índice desde cero
UPSERT_BATCH = 64

# --- Datos ---
CORPUS_DIR = Path("corpus")
KAGGLE_DATASET = "datafiniti/consumer-reviews-of-amazon-products"
N_REVIEWS = 1000                             # Muestra de reseñas a indexar (control de cuota)
RANDOM_STATE = 42
MANUAL_URL = ""                              # URL verificada del manual del fabricante (D4)
MANUAL_FILENAME = "04_manual_usuario.pdf"
MANUAL_PRODUCT = ""                          # Producto del manual; se define tras explorar el dataset

# --- Chunking ---
CHUNK_SIZE = 1000                            # Caracteres (valor inicial; se evalúa en la Sección 10)
CHUNK_OVERLAP = 150                          # ~15 %

# --- Consulta ---
TOP_K = 5
SCORE_THRESHOLD = None                       # Se calibra con evidencia en la Sección 10

CACHE_DIR = Path("cache"); CACHE_DIR.mkdir(exist_ok=True)

## 3. Carga del dataset estructurado
**Dataset:** *Consumer Reviews of Amazon Products* (Datafiniti, 2019, Kaggle).
**Justificación:** la guía incluye las reseñas como contenido indexable del pilar Ventas. Es el único dataset autorizado con texto libre, y sus productos tienen manual y garantía documentables.

### 3.1 Descarga

In [ ]:
import kagglehub
dataset_path = kagglehub.dataset_download(KAGGLE_DATASET)
csv_files = sorted(glob.glob(os.path.join(dataset_path, "*.csv")))
print("Ruta:", dataset_path)
for f in csv_files:
    print(f"- {os.path.basename(f)}  ({os.path.getsize(f)/1e6:.1f} MB)")

### 3.2 Exploración de columnas
Antes de usar cualquier campo se listan las columnas **reales** de cada archivo. No se asumen nombres.

In [ ]:
for f in csv_files:
    cols = pd.read_csv(f, nrows=0).columns.tolist()
    print(f"\n{os.path.basename(f)} ({len(cols)} columnas):\n{cols}")

### 3.3 Mapeo de campos
⚠️ **Ajustar `COLS` según la salida de la celda anterior.** Los valores escritos aquí son los nombres esperados y deben confirmarse. La celda valida que existan.

In [ ]:
COLS = {
    "product": "name",            # CONFIRMAR
    "title":   "reviews.title",   # CONFIRMAR
    "text":    "reviews.text",    # CONFIRMAR
    "rating":  "reviews.rating",  # CONFIRMAR
    "date":    "reviews.date",    # CONFIRMAR
}

selected_file = None
for f in csv_files:
    cols = set(pd.read_csv(f, nrows=0).columns)
    if set(COLS.values()).issubset(cols):
        selected_file = f
        break
assert selected_file, "Ningún CSV contiene todas las columnas de COLS. Ajustar el mapeo."
print("Archivo seleccionado:", os.path.basename(selected_file))

df_raw = pd.read_csv(selected_file, usecols=list(COLS.values()), low_memory=False)
df_raw = df_raw.rename(columns={v: k for k, v in COLS.items()})
print(df_raw.shape)
df_raw.head()

### 3.4 Tipos de datos, calidad y limitaciones

In [ ]:
print(df_raw.dtypes, "\n")
print("Nulos por columna:\n", df_raw.isna().sum(), "\n")
print("Reseñas duplicadas (texto):", df_raw["text"].duplicated().sum())
print("\nDistribución de calificaciones:\n", df_raw["rating"].value_counts(dropna=False).sort_index())
print("\nTop 10 productos por número de reseñas:")
df_raw["product"].value_counts().head(10)

### 3.5 Limpieza y muestra a indexar
Criterios: texto no nulo, longitud mínima de 30 caracteres, sin duplicados y muestra aleatoria reproducible de `N_REVIEWS`.
El producto más reseñado orienta la elección del manual D4 (`MANUAL_PRODUCT`).

> Las reseñas son **opiniones de clientes**, no especificaciones oficiales. Se etiquetan con `doc_type="review"`.

In [ ]:
df = df_raw.dropna(subset=["text"]).copy()
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() >= 30].drop_duplicates(subset=["text"])
print("Reseñas válidas:", len(df))

df_sample = df.sample(n=min(N_REVIEWS, len(df)), random_state=RANDOM_STATE).reset_index(drop=True)
print("Muestra a indexar:", len(df_sample))
df_sample.head(3)

## 4. Ingesta documental
Formatos soportados: `.md`, `.txt` y `.pdf`. En los PDF el texto se extrae página por página y se conserva el número de página.

In [ ]:
# Descarga del manual real (D4) si se definió la URL y el archivo no existe
manual_path = CORPUS_DIR / MANUAL_FILENAME
if MANUAL_URL and not manual_path.exists():
    import requests
    r = requests.get(MANUAL_URL, timeout=60)
    r.raise_for_status()
    manual_path.write_bytes(r.content)
print("Manual D4 presente:", manual_path.exists())

In [ ]:
from pypdf import PdfReader

CORPUS_META = {
    "01_politica_devoluciones_reembolsos.md": {"doc_id": "D1", "doc_type": "politica_devoluciones", "simulated": True,  "language": "es"},
    "02_politica_garantia.md":                {"doc_id": "D2", "doc_type": "politica_garantia",     "simulated": True,  "language": "es"},
    "03_terminos_servicio_sla.md":            {"doc_id": "D3", "doc_type": "terminos_servicio_sla", "simulated": True,  "language": "es"},
    MANUAL_FILENAME:                          {"doc_id": "D4", "doc_type": "manual_usuario",        "simulated": False, "language": "en", "product": MANUAL_PRODUCT},
}

def strip_front_matter(text):
    return re.sub(r"^---\n.*?\n---\n", "", text, flags=re.S)

def clean_text(text):
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

pages = []   # una entrada por página (PDF) o por archivo (MD/TXT)
for fname, meta in CORPUS_META.items():
    path = CORPUS_DIR / fname
    if not path.exists():
        print(f"⚠️ No encontrado: {fname}")
        continue
    if path.suffix.lower() == ".pdf":
        reader = PdfReader(str(path))
        for i, page in enumerate(reader.pages, start=1):
            pages.append({**meta, "source_file": fname, "page": i, "text": clean_text(page.extract_text() or "")})
    else:
        raw = path.read_text(encoding="utf-8")
        pages.append({**meta, "source_file": fname, "page": None, "text": clean_text(strip_front_matter(raw))})

df_pages = pd.DataFrame(pages)
print(df_pages.groupby("source_file")["text"].agg(["count", lambda s: s.str.len().sum()]))

### 4.1 Validación del contenido
Se revisan páginas vacías (por ejemplo, PDF escaneados sin capa de texto) y contenido duplicado.

In [ ]:
df_pages["n_chars"] = df_pages["text"].str.len()
empty = df_pages[df_pages["n_chars"] < 20]
print("Páginas vacías o casi vacías:", len(empty))

df_pages["hash"] = df_pages["text"].apply(lambda t: hashlib.md5(t.encode()).hexdigest())
print("Páginas duplicadas:", df_pages["hash"].duplicated().sum())

df_pages = df_pages[df_pages["n_chars"] >= 20].drop_duplicates(subset="hash").reset_index(drop=True)
assert set(df_pages["doc_id"]) >= {"D1", "D2", "D3"}, "Faltan documentos del corpus"
print("Documentos cargados:", sorted(df_pages["doc_id"].unique()))

In [ ]:
# Muestra del texto extraído
for _, row in df_pages.groupby("doc_id").head(1).iterrows():
    print(f"--- {row['doc_id']} | {row['source_file']} | pág {row['page']} ---\n{row['text'][:400]}\n")

## 5. Chunking
**Estrategia:** `RecursiveCharacterTextSplitter`, que corta primero por títulos y párrafos y después por oraciones. Así preserva unidades semánticas completas, como cláusulas o pasos de un procedimiento.

**Valores iniciales y justificación:**
- `CHUNK_SIZE = 1000` caracteres: una cláusula de política suele ocupar 1–2 párrafos. Este tamaño la contiene completa sin mezclar demasiados temas en un solo vector.
- `CHUNK_OVERLAP = 150` (~15 %): evita que una condición quede partida entre dos fragmentos.
- **Reseñas: 1 reseña = 1 fragmento**, porque ya son unidades semánticas cortas e independientes.

Estos valores son un punto de partida. En la Sección 10 se comparan 500 / 1000 / 1500 con el conjunto de preguntas de prueba.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def build_doc_chunks(df_pages, chunk_size, chunk_overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n## ", "\n### ", "\n\n", "\n", ". ", " ", ""],
    )
    rows = []
    for _, p in df_pages.iterrows():
        for i, piece in enumerate(splitter.split_text(p["text"])):
            has_page = pd.notna(p["page"])
            page_tag = f"p{int(p['page'])}" if has_page else "p0"
            rows.append({
                "chunk_id": f"{p['doc_id']}_{page_tag}_c{i}",
                "doc_id": p["doc_id"], "source_file": p["source_file"], "doc_type": p["doc_type"],
                "simulated": bool(p["simulated"]), "language": p["language"],
                "page": int(p["page"]) if has_page else None,
                "chunk_index": i, "product": p.get("product") or None, "rating": None,
                "text": piece.strip(),
            })
    return rows

def build_review_chunks(df_sample):
    rows = []
    for i, r in df_sample.iterrows():
        title = str(r["title"]).strip() if pd.notna(r["title"]) else ""
        text = f"{title}. {r['text']}" if title else r["text"]
        rows.append({
            "chunk_id": f"R_{i}", "doc_id": "DS", "source_file": "consumer_reviews_amazon.csv",
            "doc_type": "review", "simulated": False, "language": "en", "page": None, "chunk_index": 0,
            "product": str(r["product"]), "rating": None if pd.isna(r["rating"]) else float(r["rating"]),
            "text": text,
        })
    return rows

chunks = build_doc_chunks(df_pages, CHUNK_SIZE, CHUNK_OVERLAP) + build_review_chunks(df_sample)
df_chunks = pd.DataFrame(chunks)
assert df_chunks["chunk_id"].is_unique
print("Total de fragmentos:", len(df_chunks))
df_chunks.assign(n_chars=df_chunks["text"].str.len()).groupby("doc_id")["n_chars"].describe()

In [ ]:
# Ejemplos de fragmentos por documento
for _, row in df_chunks.groupby("doc_id").head(1).iterrows():
    print(f"[{row['chunk_id']}] ({len(row['text'])} chars)\n{row['text'][:500]}\n")

## 6. Embeddings
**Modelo:** Gemini Embedding 2 (Google AI Studio). Es el modelo señalado en el Paso 5 de la guía, es multilingüe y permite ajustar la dimensión de salida.
- **Nota técnica:** la guía menciona `task_type=RETRIEVAL_QUERY`. Según la documentación oficial vigente de Google, `gemini-embedding-2` **no acepta el campo `task_type`**: la tarea se indica con un prefijo dentro del texto. El efecto es el mismo (recuperación asimétrica documento ↔ consulta):
  - Documentos (equivale a `RETRIEVAL_DOCUMENT`): `title: none | text: {contenido}`.
  - Consultas (equivale a `RETRIEVAL_QUERY`): `task: question answering | query: {pregunta}`.
- Dimensión: 768 (el modelo renormaliza automáticamente al truncar).
- Cada texto se envía como un `Content` independiente para obtener un vector por texto. Si se envían como partes de un mismo contenido, el modelo devuelve un único vector agregado.

Controles de cuota: lotes de `EMBED_BATCH`, ritmo limitado por tokens/minuto (`EMBED_TPM_LIMIT`), reintentos con espera exponencial y caché local para no volver a vectorizar textos ya procesados.

In [ ]:
from google import genai
from google.genai import types

gclient = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

def _format(text, task_type):
    # task_type se conserva como etiqueta legible; se traduce al prefijo que exige gemini-embedding-2
    if task_type == "RETRIEVAL_QUERY":
        return QUERY_PREFIX + text
    return DOC_TEMPLATE.format(text=text)

def _embed_call(texts, task_type):
    # Un Content por texto → un vector por texto
    contents = [types.Content(parts=[types.Part(text=_format(t, task_type))]) for t in texts]
    resp = gclient.models.embed_content(
        model=EMBED_MODEL,
        contents=contents,
        config=types.EmbedContentConfig(output_dimensionality=EMBED_DIM),
    )
    return [e.values for e in resp.embeddings]

def embed_texts(texts, task_type, batch_size=EMBED_BATCH, max_retries=5):
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        est_tokens = sum(len(t) for t in batch) / 4          # estimación aproximada: ~4 caracteres por token
        for attempt in range(max_retries):
            try:
                vecs = _embed_call(batch, task_type)
                break
            except Exception as e:
                wait = 15 * 2 ** attempt
                print(f"Lote {start}: error {type(e).__name__}: {str(e)[:120]} → reintento en {wait}s")
                time.sleep(wait)
        else:
            raise RuntimeError(f"Falló el lote que inicia en {start}")
        if len(vecs) != len(batch):
            raise RuntimeError(f"El modelo devolvió {len(vecs)} vectores para {len(batch)} textos (lote {start})")
        vectors.extend(vecs)
        # Ritmo: no superar EMBED_TPM_LIMIT tokens por minuto
        time.sleep(max(EMBED_PAUSE_S, 60 * est_tokens / EMBED_TPM_LIMIT))
        print(f"Vectorizados {min(start + batch_size, len(texts))}/{len(texts)}", end="\r")
    return np.asarray(vectors, dtype=np.float32)

In [ ]:
# Prueba mínima antes del proceso masivo
test_vec = embed_texts(["prueba de conectividad del modelo de embeddings"], "RETRIEVAL_DOCUMENT")
print("Forma:", test_vec.shape, "| Norma L2:", float(np.linalg.norm(test_vec[0])))
assert test_vec.shape == (1, EMBED_DIM)

In [ ]:
cache_file = CACHE_DIR / f"emb_{EMBED_MODEL.replace('/', '_')}_{EMBED_DIM}_{CHUNK_SIZE}_{CHUNK_OVERLAP}.npz"

if cache_file.exists():
    data = np.load(cache_file, allow_pickle=True)
    assert list(data["ids"]) == df_chunks["chunk_id"].tolist(), "La caché no coincide con los fragmentos actuales"
    doc_vectors = data["vectors"]
    print("Embeddings cargados desde caché:", cache_file)
else:
    t0 = time.time()
    doc_vectors = embed_texts(df_chunks["text"].tolist(), "RETRIEVAL_DOCUMENT")
    print(f"\nTiempo: {time.time() - t0:.1f}s")
    np.savez(cache_file, ids=np.array(df_chunks["chunk_id"].tolist(), dtype=object), vectors=doc_vectors)

### 6.1 Validación de los vectores

In [ ]:
norms = np.linalg.norm(doc_vectors, axis=1)
print("Forma:", doc_vectors.shape)
print("NaN:", int(np.isnan(doc_vectors).sum()), "| Inf:", int(np.isinf(doc_vectors).sum()))
print(f"Norma L2 → min {norms.min():.4f} | media {norms.mean():.4f} | max {norms.max():.4f}")
assert doc_vectors.shape == (len(df_chunks), EMBED_DIM)
assert not np.isnan(doc_vectors).any()

## 7. Configuración cloud (Qdrant Cloud)
- **Servicio:** Qdrant Cloud, capa gratuita (clúster de un nodo).
- **Métrica:** `Cosine`. Mide el ángulo entre vectores e ignora su magnitud; es la métrica estándar para embeddings semánticos. Qdrant normaliza los vectores al almacenarlos.
- **Índice de payload** en `doc_type` para permitir filtrar por tipo de documento.

In [ ]:
from qdrant_client import QdrantClient, models

qclient = QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ["QDRANT_API_KEY"], timeout=60)

# Prueba de conectividad
print("Colecciones existentes:", [c.name for c in qclient.get_collections().collections])

In [ ]:
if RECREATE_COLLECTION and qclient.collection_exists(COLLECTION):
    qclient.delete_collection(COLLECTION)
    print("Colección eliminada para reconstrucción")

if not qclient.collection_exists(COLLECTION):
    qclient.create_collection(
        collection_name=COLLECTION,
        vectors_config=models.VectorParams(size=EMBED_DIM, distance=models.Distance.COSINE),
    )
    print("Colección creada:", COLLECTION)

qclient.create_payload_index(COLLECTION, field_name="doc_type", field_schema=models.PayloadSchemaType.KEYWORD)

info = qclient.get_collection(COLLECTION)
print("Dimensión:", info.config.params.vectors.size, "| Distancia:", info.config.params.vectors.distance)
assert info.config.params.vectors.size == EMBED_DIM

## 8. Indexación (batch upsert)
Cada punto tiene un ID determinista (UUID v5 derivado de `chunk_id`). Así, una reejecución **actualiza** los puntos en lugar de duplicarlos. El payload guarda el texto y los metadatos.

In [ ]:
def to_point(row, vector):
    payload = {k: (None if (isinstance(v, float) and np.isnan(v)) else v) for k, v in row.items()}
    return models.PointStruct(id=str(uuid.uuid5(uuid.NAMESPACE_URL, "aca1-ventas/" + row["chunk_id"])), vector=vector.tolist(), payload=payload)

records = df_chunks.to_dict(orient="records")
errors = []
t0 = time.time()
for start in range(0, len(records), UPSERT_BATCH):
    batch = [to_point(r, doc_vectors[start + i]) for i, r in enumerate(records[start:start + UPSERT_BATCH])]
    try:
        qclient.upsert(collection_name=COLLECTION, points=batch, wait=True)
    except Exception as e:
        errors.append((start, str(e)[:200]))
        print(f"Error en lote {start}: {str(e)[:200]}")
print(f"Upsert terminado en {time.time() - t0:.1f}s | lotes con error: {len(errors)}")

### 8.1 Verificación de persistencia

In [ ]:
count = qclient.count(COLLECTION, exact=True).count
print("Puntos en la colección:", count, "| Fragmentos locales:", len(df_chunks))
assert count == len(df_chunks), "La cantidad de puntos no coincide"

for dt in df_chunks["doc_type"].unique():
    n = qclient.count(COLLECTION, count_filter=models.Filter(must=[models.FieldCondition(key="doc_type", match=models.MatchValue(value=dt))]), exact=True).count
    print(f"  {dt}: {n}")

## 9. Consultas semánticas
La función `buscar()` sigue estos pasos:
1. Vectoriza la pregunta en formato consulta (equivalente a `RETRIEVAL_QUERY`).
2. Consulta Qdrant Cloud.
3. Devuelve el Top-k con score, fuente, tipo de documento y fragmento.

Si `SCORE_THRESHOLD` está calibrado y el mejor score queda por debajo, informa que no hay respaldo suficiente.

> El score es una **similitud coseno** (rango teórico −1 a 1). **No es una probabilidad** de que la respuesta sea correcta.

In [ ]:
def buscar(pregunta, k=TOP_K, doc_type=None, mostrar=True):
    qvec = embed_texts([pregunta], "RETRIEVAL_QUERY")[0]
    flt = None
    if doc_type:
        flt = models.Filter(must=[models.FieldCondition(key="doc_type", match=models.MatchValue(value=doc_type))])
    hits = qclient.query_points(COLLECTION, query=qvec.tolist(), limit=k, query_filter=flt, with_payload=True).points
    rows = [{
        "rank": i + 1, "score": round(h.score, 4), "doc_id": h.payload.get("doc_id"),
        "doc_type": h.payload.get("doc_type"), "source_file": h.payload.get("source_file"),
        "page": h.payload.get("page"), "chunk_id": h.payload.get("chunk_id"),
        "simulado": h.payload.get("simulated"), "fragmento": h.payload.get("text", "")[:300],
    } for i, h in enumerate(hits)]
    res = pd.DataFrame(rows)
    if mostrar:
        print(f"Pregunta: {pregunta}")
        if SCORE_THRESHOLD is not None and (res.empty or res["score"].max() < SCORE_THRESHOLD):
            print("⚠️ No hay información suficiente en el corpus para responder con respaldo.")
        display(res)
    return res

In [ ]:
_ = buscar("¿Cuáles son las condiciones para devolver un producto?")

In [ ]:
_ = buscar("¿Qué cubre la garantía?", doc_type="politica_garantia")

### 9.1 Verificación del cálculo de similitud coseno
Se recalcula el coseno localmente entre la consulta y los vectores almacenados, y se compara con el score que devuelve Qdrant.

In [ ]:
q = "¿Qué pasos debe seguir un cliente para solicitar un reembolso?"
qv = embed_texts([q], "RETRIEVAL_QUERY")[0]
hits = qclient.query_points(COLLECTION, query=qv.tolist(), limit=3, with_vectors=True).points
qn = qv / np.linalg.norm(qv)
for h in hits:
    dv = np.asarray(h.vector, dtype=np.float32)
    local_cos = float(np.dot(qn, dv / np.linalg.norm(dv)))
    print(f"{h.payload['chunk_id']:<15} Qdrant={h.score:.4f}  local={local_cos:.4f}  dif={abs(h.score - local_cos):.2e}")

## 10. Pruebas
### 10.1 Conjunto de preguntas de prueba
Tomado de `corpus/README_corpus.md`. `expected` indica el documento que debería respaldar la respuesta; `None` corresponde a preguntas sin respaldo.

In [ ]:
TEST_SET = [
    ("Q01", "¿Cuántos días tengo para retractarme de una compra?", "respondible", ["D1"]),
    ("Q02", "¿En cuánto tiempo me devuelven el dinero si ejerzo el retracto?", "respondible", ["D1"]),
    ("Q03", "¿Qué pasos debo seguir para solicitar un reembolso?", "respondible", ["D1"]),
    ("Q04", "Me llegó el producto roto, ¿qué hago?", "respondible", ["D1"]),
    ("Q05", "¿Cuánto dura la garantía de un lector electrónico?", "respondible", ["D2"]),
    ("Q06", "¿La garantía cubre si se me cae el dispositivo al agua?", "respondible", ["D2"]),
    ("Q07", "¿Qué pasa si la misma falla se repite después de la reparación?", "respondible", ["D2"]),
    ("Q08", "¿Cuánto tarda un envío a Bogotá?", "respondible", ["D3"]),
    ("Q09", "¿Puedo cancelar un pedido que ya fue despachado?", "respondible", ["D3"]),
    ("Q10", "¿En cuánto tiempo responden una queja o reclamo?", "respondible", ["D3"]),
    ("Q11", "¿Cómo cargo la batería del dispositivo?", "respondible", ["D4"]),          # AJUSTAR al manual
    ("Q12", "¿Qué opinan los clientes sobre la duración de la batería?", "respondible", ["DS"]),
    ("Q13", "Tengo un problema con mi pedido", "ambigua", ["D1", "D3"]),
    ("Q14", "¿Cuánto tiempo tengo?", "ambigua", ["D1", "D2"]),
    ("Q15", "¿Venden teléfonos celulares?", "sin_respaldo", None),
    ("Q16", "¿Ofrecen financiación a cuotas sin intereses?", "sin_respaldo", None),
    ("Q17", "¿Cuál es la dirección de la tienda física?", "sin_respaldo", None),
]

### 10.2 Ejecución y métricas de recuperación
- **Hit@1 / Hit@k:** el documento esperado aparece en la posición 1 o dentro del Top-k.
- **MRR:** promedio del inverso de la posición del primer acierto.
- Para las preguntas sin respaldo se registra el score máximo. Esto sirve para calibrar `SCORE_THRESHOLD`.

In [ ]:
def evaluar(test_set, k=TOP_K):
    rows = []
    for qid, q, tipo, expected in test_set:
        res = buscar(q, k=k, mostrar=False)
        top_ids = res["doc_id"].tolist()
        first_hit = next((i + 1 for i, d in enumerate(top_ids) if expected and d in expected), None)
        rows.append({
            "id": qid, "tipo": tipo, "pregunta": q, "esperado": expected,
            "top1_doc": top_ids[0] if top_ids else None, "top1_score": res["score"].iloc[0] if len(res) else None,
            "hit@1": (first_hit == 1) if expected else None,
            f"hit@{k}": (first_hit is not None) if expected else None,
            "rr": (1 / first_hit if first_hit else 0.0) if expected else None,
        })
    return pd.DataFrame(rows)

df_eval = evaluar(TEST_SET)
display(df_eval)

resp = df_eval[df_eval["tipo"] != "sin_respaldo"]
print(f"Hit@1: {resp['hit@1'].mean():.2%} | Hit@{TOP_K}: {resp[f'hit@{TOP_K}'].mean():.2%} | MRR: {resp['rr'].mean():.3f}")
print("\nScore top-1 por tipo de pregunta:")
print(df_eval.groupby("tipo")["top1_score"].describe()[["mean", "min", "max"]])

### 10.3 Calibración del umbral de "información insuficiente"
El umbral se fija **a partir de la evidencia**: un valor entre el score máximo de las preguntas sin respaldo y el score mínimo de las respondibles con acierto. Si los rangos se solapan, se documenta como limitación y no se fuerza un umbral.

In [ ]:
max_neg = df_eval.loc[df_eval["tipo"] == "sin_respaldo", "top1_score"].max()
min_pos = df_eval.loc[(df_eval["tipo"] == "respondible") & (df_eval["hit@1"] == True), "top1_score"].min()
print(f"Máx. score sin respaldo: {max_neg:.4f} | Mín. score respondible acertada: {min_pos:.4f}")
if min_pos > max_neg:
    SCORE_THRESHOLD = round((min_pos + max_neg) / 2, 4)
    print("Umbral calibrado:", SCORE_THRESHOLD)
else:
    print("Los rangos se solapan: no se fija umbral. Se reporta como limitación.")

### 10.4 Experimento de tamaño de chunk (opcional, consume cuota)
Compara 500 / 1000 / 1500 caracteres usando una **colección temporal** que solo contiene documentos (sin reseñas). Hay que verificar la cuota disponible antes de ejecutar.

In [ ]:
RUN_CHUNK_EXPERIMENT = False

if RUN_CHUNK_EXPERIMENT:
    doc_tests = [t for t in TEST_SET if t[3] and set(t[3]) <= {"D1", "D2", "D3", "D4"}]
    results = []
    for size in [500, 1000, 1500]:
        tmp = f"{COLLECTION}_exp_{size}"
        ch = pd.DataFrame(build_doc_chunks(df_pages, size, int(size * 0.15)))
        vecs = embed_texts(ch["text"].tolist(), "RETRIEVAL_DOCUMENT")
        if qclient.collection_exists(tmp):
            qclient.delete_collection(tmp)
        qclient.create_collection(tmp, vectors_config=models.VectorParams(size=EMBED_DIM, distance=models.Distance.COSINE))
        qclient.upsert(tmp, points=[to_point(r, vecs[i]) for i, r in enumerate(ch.to_dict("records"))], wait=True)
        hits1 = 0
        for _, q, _, exp in doc_tests:
            qv = embed_texts([q], "RETRIEVAL_QUERY")[0]
            top = qclient.query_points(tmp, query=qv.tolist(), limit=1, with_payload=True).points
            hits1 += int(bool(top) and top[0].payload["doc_id"] in exp)
        results.append({"chunk_size": size, "n_chunks": len(ch), "hit@1": hits1 / len(doc_tests)})
        qclient.delete_collection(tmp)
    display(pd.DataFrame(results))

### 10.5 Matriz de pruebas técnicas
`Estado` se actualiza **solo** con evidencia de ejecución. Valor inicial: *Pendiente de ejecución*.

| ID | Requisito | Objetivo | Procedimiento | Resultado esperado | Resultado obtenido | Estado | Evidencia |
|---|---|---|---|---|---|---|---|
| T01 | Conectividad | Conectar con Qdrant Cloud | Celda 7 `get_collections()` | Sin error | | Pendiente de ejecución | |
| T02 | Extracción | Texto de MD y PDF | Sección 4 | 0 páginas vacías relevantes | | Pendiente de ejecución | |
| T03 | Chunking | Fragmentos coherentes | Sección 5 | IDs únicos, tamaños ≤ CHUNK_SIZE | | Pendiente de ejecución | |
| T04 | Embeddings | Vectores válidos | Sección 6.1 | Sin NaN/Inf | | Pendiente de ejecución | |
| T05 | Dimensión | Compatibilidad vector–índice | Secciones 6.1 y 7 | 768 = 768 | | Pendiente de ejecución | |
| T06 | Indexación por lotes | Upsert sin errores | Sección 8 | 0 lotes con error | | Pendiente de ejecución | |
| T07 | Persistencia | Datos en la nube | Sección 8.1 | count = n fragmentos | | Pendiente de ejecución | |
| T08 | Top-k | Recuperación consistente | Sección 10.2 | Hit@k ≥ objetivo definido | | Pendiente de ejecución | |
| T09 | Fuentes | Trazabilidad | Sección 9 | Cada resultado con fuente | | Pendiente de ejecución | |
| T10 | Sin respaldo | Detección de preguntas sin respaldo | Sección 10.3 | Umbral calibrado o limitación documentada | | Pendiente de ejecución | |
| T11 | Errores | Reintentos y manejo de fallos | Secciones 6 y 8 | Fallos registrados sin detener el proceso | | Pendiente de ejecución | |
| T12 | Reproducibilidad | Ejecución secuencial | *Restart & Run all* | Sin errores | | Pendiente de ejecución | |
| T13 | Seguridad | Sin credenciales en el código | Revisión del .ipynb y del repositorio | 0 claves expuestas | | Pendiente de ejecución | |

## 11. Conclusiones
> **Pendiente:** se redactan solo con los resultados reales obtenidos en las Secciones 8 a 10.

- **Resultados reales:** —
- **Cumplimiento de objetivos:** —
- **Limitaciones:**
  - Los documentos D1–D3 son simulados.
  - Las reseñas y el manual están en inglés y las consultas en español (recuperación translingüe).
  - El modelo de embeddings está en versión preview.
  - El clúster gratuito no tiene alta disponibilidad.
- **Mejoras futuras:**
  - Generación de respuestas con un LLM (RAG completo).
  - Reranking.
  - Búsqueda híbrida (densa + léxica).
  - Ampliar el corpus.

### Referencias (APA 7)
- Datafiniti. (2019). *Consumer reviews of Amazon products* [Conjunto de datos]. Kaggle. https://www.kaggle.com/datasets/datafiniti/consumer-reviews-of-amazon-products
- Lewis, P., Perez, E., Piktus, A., Petroni, F., Karpukhin, V., Goyal, N., Küttler, H., Lewis, M., Yih, W., Rocktäschel, T., Riedel, S., & Kiela, D. (2020). Retrieval-augmented generation for knowledge-intensive NLP tasks. *Advances in Neural Information Processing Systems, 33*, 9459–9474. https://arxiv.org/abs/2005.11401
- Qdrant Solutions GmbH. (2024). *Qdrant documentation*. https://qdrant.tech/documentation/
- *(Completar: Gemini Embedding 2 – documentación de Google; manual D4; Ley 1480 de 2011; Decreto 587 de 2016.)*